# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 1: KHÁM PHÁ DỮ LIỆU & LÀM SẠCH VĂN BẢN (DATA CLEANING & EDA)

> **Mục tiêu:**
> - Khám phá cấu trúc dữ liệu.
> - Tìm hiểu đặc điểm email thường và thư rác.
> - Lột bỏ lớp ngụy trang của spam và xóa sạch các thư trùng lặp để mô hình học chuẩn xác nhất.

---
## 0. GIỚI THIỆU BỘ DỮ LIỆU CEAS_08
- **Quy mô thô:** 39.154 email (khoảng 65 MB).
- **Biến mục tiêu:** `label` (0: Ham - Thư thường, 1: Spam - Thư rác).
- **Các thuộc tính:** `sender`, `receiver`, `date`, `subject`, `body`, `label`, `urls`.

---
## Mục 1: Khám phá dữ liệu tổng quan — Cấu trúc, Chất lượng & Đơn biến (Overview EDA)

### 1.1 Khai báo Thư viện & Cấu hình Thực nghiệm
**Mục tiêu:** Nạp thư viện cốt lõi và cố định `RANDOM_SEED = 42` bảo đảm tính tái lập kết quả.

In [ ]:
# Khai báo thư viện & Thiết lập môi trường
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

print("Môi trường phân tích sẵn sàng (RANDOM_SEED = 42).")

### 1.2 Khảo sát Cấu trúc & Đánh giá Sơ bộ Chất lượng Dữ liệu
**Mục tiêu:** Nạp dữ liệu thô, kiểm tra quy mô, kiểu dữ liệu, khuyết thiếu và trùng lặp kỹ thuật.

In [ ]:
# Khảo sát cấu trúc & Đánh giá chất lượng dữ liệu thô
data_path = os.path.join('..', 'data', 'raw', 'CEAS_08.csv')
df = pd.read_csv(data_path)

print(f"1. Quy mô dữ liệu thô: {df.shape[0]:,} dòng, {df.shape[1]} cột")
print("\n2. Giá trị khuyết thiếu tổng quát trên các cột:")
print(df.isnull().sum())
print(f"\n3. Trùng lặp thô trên toàn bộ 7 cột: {df.duplicated().sum()} dòng")
print(f"4. Trùng lặp thô trên cặp (subject, body): {df.duplicated(subset=['subject', 'body']).sum()} dòng")
display(df.head(5))

**Đúc kết khảo sát sơ bộ:**
* **Dữ liệu rất đầy đủ:** Chỉ thiếu một ít ở cột người nhận (462 dòng) và tiêu đề (28 dòng); nội dung và nhãn thư có đủ 100%.
* **Thoạt nhìn không có thư trùng:** Khi so khớp toàn bộ cột hoặc cặp tiêu đề - nội dung thô, tỷ lệ trùng bằng 0. Dữ liệu ban đầu trông rất sạch để bắt đầu phân tích.

### 1.3 Phân tích Đơn biến (Univariate Analysis)
**Mục tiêu:** Khảo sát phân bố biến mục tiêu `label`, sự xuất hiện của `urls`, độ dài tiêu đề và top tên miền người gửi.

In [ ]:
# Phân tích đơn biến 4 thuộc tính
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Biến mục tiêu
df['label'].value_counts().plot.pie(
    labels=['Spam (1)', 'Ham (0)'], autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
    ax=axes[0, 0], startangle=90, explode=(0.03, 0)
)
axes[0, 0].set_title("Tỷ lệ Spam vs Ham (Dữ liệu thô)", fontweight='bold')
axes[0, 0].set_ylabel('')

# 2. Thuộc tính URLs
sns.countplot(data=df, x='urls', ax=axes[0, 1], palette=['#3498db', '#f39c12'])
axes[0, 1].set_title("Sự hiện diện của liên kết URLs", fontweight='bold')

# 3. Độ dài tiêu đề
subj_len = df['subject'].fillna('').str.len()
sns.histplot(subj_len[subj_len < 120], bins=30, kde=True, ax=axes[1, 0], color='#9b59b6')
axes[1, 0].set_title("Phân phối Độ dài Tiêu đề (< 120 ký tự)", fontweight='bold')

# 4. Top 10 Tên miền người gửi
domains = df['sender'].astype(str).apply(lambda x: x.split('@')[-1].lower() if '@' in str(x) else 'unknown')
domains.value_counts().head(10).plot.barh(ax=axes[1, 1], color='#1abc9c').invert_yaxis()
axes[1, 1].set_title("Top 10 Tên miền Người gửi", fontweight='bold')

plt.tight_layout()
plt.show()

**Đúc kết phân tích đơn biến:**
* **Tỷ lệ thư khá cân bằng:** Số lượng thư rác và thư thường xấp xỉ nhau, không bị lệch quá nhiều nên không cần dùng kỹ thuật bù trừ mẫu.
* **Đặc điểm chung:** Phần lớn email đều có chứa đường link (URL); tiêu đề thư đa số khá ngắn; người gửi nhiều nhất chủ yếu đến từ tên miền kỹ thuật mã nguồn mở.

---
## Mục 2: Khám phá dữ liệu chuyên sâu — Đa biến, Ngoại lai & Đúc kết định hướng (In-depth EDA)

### 2.1 Khảo sát Đa biến: So sánh Độ dài Văn bản giữa Thư Rác và Thư Thường
**Mục tiêu:** Sử dụng Boxplot so sánh phân phối độ dài tiêu đề và nội dung theo nhãn.

In [ ]:
# So sánh độ dài văn bản theo nhãn
df['subject_len'] = df['subject'].fillna('').str.len()
df['body_len'] = df['body'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df, x='label', y='subject_len', ax=axes[0], palette=['#2ecc71', '#e74c3c'], showfliers=False)
axes[0].set_xticklabels(['Ham (0)', 'Spam (1)'])
axes[0].set_title("Độ dài Tiêu đề theo Nhãn", fontweight='bold')

sns.boxplot(data=df, x='label', y=np.log1p(df['body_len']), ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_xticklabels(['Ham (0)', 'Spam (1)'])
axes[1].set_title("Log(Độ dài Nội dung + 1) theo Nhãn", fontweight='bold')
plt.tight_layout()
plt.show()

**Đúc kết:**
- Thư thường có trung vị tiêu đề dài hơn (45 ký tự) so với thư rác (27 ký tự).
- Nội dung email lệch phải rất mạnh, cần biến đổi log để thu hẹp khoảng cách này cho mô hình dễ học

### 2.2 Khám phá Bản chất Từ vựng: Đối lập giữa Thư Ham và Spam
**Mục tiêu:** Trích xuất Top từ khóa trong tiêu đề để phát hiện bản chất thực tế của từng nhóm email.

In [ ]:
# Trích xuất Top 10 từ khóa tiêu đề của Ham và Spam
import re
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP = set(ENGLISH_STOP_WORDS)

def extract_keywords(series, top_n=10):
    words = []
    for text in series.dropna().astype(str):
        tokens = [w for w in re.sub(r'[^a-zA-Z\s]', ' ', text).lower().split() if w not in STOP and len(w) > 2]
        words.extend(tokens)
    return Counter(words).most_common(top_n)

top_ham = pd.DataFrame(extract_keywords(df[df['label']==0]['subject']), columns=['Từ khóa Ham', 'Tần suất'])
top_spam = pd.DataFrame(extract_keywords(df[df['label']==1]['subject']), columns=['Từ khóa Spam', 'Tần suất'])
display(pd.concat([top_ham, top_spam], axis=1))

**Đúc kết từ vựng giữa 2 loại thư:**
* **Thư thường (Ham):** Toàn là từ ngữ công việc và lập trình kỹ thuật (như `python`, `opensuse`, `dev`, `bug`, `patch`).
* **Thư rác (Spam):** Tập trung vào 3 nhóm lừa đảo phổ biến: Giả mạo tin tức/cảnh báo (`cnn daily`, `alert`), hàng hiệu giả (`replica`, `rolex`, `watches`), và thuốc tăng cường sinh lý (`penis`).

### 2.3 Điều tra Hành vi Ngụy trang & Chiến dịch Spam Hàng loạt
**Mục tiêu:** Điều tra các chiến dịch gửi email lặp lại (`sender` + `subject`) và dấu hiệu hội thoại (`Re:`).

In [ ]:
# Điều tra hành vi phát tán lặp lại và tiền tố trao đổi Re:
dup_mask = df.duplicated(subset=['sender', 'subject'], keep='first')
dup_ham = (dup_mask & (df['label'] == 0)).sum()
dup_spam = (dup_mask & (df['label'] == 1)).sum()

subj_lower = df['subject'].fillna('').str.lower()
# Dùng r'\bre:' vì dấu ':' là ký tự phi từ ngữ (\W), \b đứng sau nó sẽ không khớp nếu theo sau là khoảng trắng
has_re = subj_lower.str.contains(r'\bre:')
re_ham = (has_re & (df['label'] == 0)).sum() / (df['label'] == 0).sum() * 100
re_spam = (has_re & (df['label'] == 1)).sum() / (df['label'] == 1).sum() * 100

print(f"1. Trùng lặp cặp (sender, subject): {dup_mask.sum():,} mẫu (Ham: {dup_ham:,} [{dup_ham/dup_mask.sum()*100:.1f}%] vs Spam: {dup_spam:,} [{dup_spam/dup_mask.sum()*100:.1f}%])")
print(f"2. Tỷ lệ mang tiền tố đối thoại 'Re:': Thư thường = {re_ham:.1f}% vs Thư rác = {re_spam:.1f}%")

**Đúc kết hành vi & Phát hiện then chốt:**
* **Thư có chữ `Re:` hầu hết là thư thật:**  
  Tiền tố `Re:` (thư trả lời) xuất hiện ở email thường nhiều gấp hơn 15 lần so với thư rác. Điều này thể hiện sự tương tác qua lại của người thật trong công việc, điều mà thư rác hiếm khi có.

* **Nhiều thư gửi từ cùng một địa chỉ không phải là thư rác:**  
  Email trùng lặp người gửi và tiêu đề tưởng là gửi rác hàng loạt, nhưng thực tế hơn 95% là thư bình thường (như thông báo tự động từ hệ thống quản lý lỗi, nhóm làm việc mã nguồn mở...). Ngược lại, kẻ phát tán thư rác không bao giờ dùng một địa chỉ cố định mà liên tục thay đổi.

* **Kẻ gửi thư rác ngụy trang để "qua mặt" bộ lọc:**  
  Để gửi hàng triệu bức thư mà không bị hệ thống quét phát hiện trùng lặp, kẻ gửi spam chèn thêm các đoạn mã HTML ngẫu nhiên và thay đổi đuôi đường dẫn (URL) trong từng thư. Cách này khiến hệ thống máy tính tưởng mỗi email là một nội dung hoàn toàn mới.

### 2.4 Kiểm tra Ngoại lai & Tương quan URL
**Mục tiêu:** Kiểm tra ngoại lai theo IQR và phân tích hiện tượng email siêu dài.

In [ ]:
# Đánh giá ngoại lai IQR và tương quan URL
q1 = df['body_len'].quantile(0.25)
q3 = df['body_len'].quantile(0.75)
iqr_outliers = (df['body_len'] > q3 + 1.5 * (q3 - q1)).sum()
print(f"Email ngoại lai siêu dài (> 3.771 ký tự): {iqr_outliers:,} ({iqr_outliers/len(df)*100:.2f}%)")

# Tỷ lệ Spam theo URL
url_rate = df.groupby('urls')['label'].mean() * 100
print(f"Tỷ lệ Spam: Không có URL={url_rate[0]:.2f}%, Có URL={url_rate[1]:.2f}% (Chênh lệch: {url_rate[1]-url_rate[0]:.2f}%)")

**Đúc kết về thư siêu dài và đường link (URL):**
* **Thư có độ dài bất thường (chiếm ~14.4%):** Kẻ gửi spam cố tình chèn thêm rất nhiều đoạn văn dài hoặc từ ngữ linh tinh để làm loãng các từ khóa rác nhằm đánh lừa bộ lọc (*Bayesian Poisoning*). Không nên xóa các thư này, mà cần dùng hàm Log (`log1p`) và chuẩn hóa TF-IDF để mô hình không bị lệch.
* **Đường link (URL) chưa đủ để nhận diện:** Cả thư thường lẫn thư rác đều chứa link với tỷ lệ gần như nhau (chênh lệch chưa tới 3%). Vì vậy, việc bắt thư rác phải dựa chính vào nội dung chữ viết thay vì chỉ nhìn vào việc có link hay không.

### 2.5 Tổng kết EDA & Định hướng Hành động Kỹ thuật (Actionable Insights)
**Mục tiêu:** Đúc kết bằng chứng khám phá thành kế hoạch hành động cụ thể.

In [ ]:
# Bảng đúc kết EDA định hướng hành động
eda_summary = pd.DataFrame([
    {"Phát hiện EDA": "28 email khuyết subject", "Hành động kỹ thuật": "Điền chuỗi rỗng ''"},
    {"Phát hiện EDA": "Spammer ngụy trang HTML/URL che giấu thư rác gửi loạt", "Hành động kỹ thuật": "Làm sạch NLP rồi khử trùng text_clean"},
    {"Phát hiện EDA": "Từ vựng phân tách rõ (python/bug vs replica/cnn)", "Hành động kỹ thuật": "Dùng TF-IDF n-gram (1,2) + Chi-Square"},
    {"Phát hiện EDA": "Độ dài lệch phải nặng (14.36% ngoại lai)", "Hành động kỹ thuật": "Nén log1p và chuẩn hóa L2"}
])
display(eda_summary)

**Đúc kết định hướng:** Toàn bộ quá trình EDA ở Mục 1 và Mục 2 đã làm tìm ra các phát hiện gợi ý cho các bước hành động tiếp theo

---
## Mục 3: Tiền xử lý dữ liệu & Làm sạch văn bản (Data Cleaning)

### 3.1 Xử lý Giá trị Khuyết thiếu Tiêu đề, Nội dung & Người nhận
**Mục tiêu:** Điền chuỗi rỗng cho `subject` và `body`, điền giá trị tường minh `'unknown'` cho `receiver`, bảo đảm tính toán ổn định và an toàn kiểu dữ liệu.

In [ ]:
# Xử lý giá trị khuyết thiếu
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
df['receiver'] = df['receiver'].fillna('unknown')
print("Xử lý xong giá trị khuyết thiếu:")
print(f"- subject còn khuyết: {df['subject'].isnull().sum()} dòng")
print(f"- body còn khuyết: {df['body'].isnull().sum()} dòng")
print(f"- receiver còn khuyết: {df['receiver'].isnull().sum()} dòng")

### 3.2 Xây dựng Hàm Tiền xử lý Văn bản NLP (`clean_text`) & Khử Trùng lặp Nội dung (Zero Duplicate Leakage)
**Mục tiêu:** Xây dựng hàm `clean_text` lột sạch lớp vỏ HTML, URL và từ dừng, sau đó khử trùng lặp theo nội dung sạch.

In [ ]:
# Hàm clean_text và khử trùng lặp nội dung
import re
import html
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP_WORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http[s]?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    words = [w for w in text.lower().split() if w not in STOP_WORDS and len(w) > 1]
    return ' '.join(words)

df['text_clean'] = (df['subject'] + ' ' + df['body']).apply(clean_text)

# Khử trùng lặp nội dung văn bản sạch
before = len(df)
df = df.drop_duplicates(subset=['text_clean']).reset_index(drop=True)
print(f"Đã loại bỏ {before - len(df):,} mẫu trùng lặp nội dung.")
print(f"Dữ liệu sạch còn lại: {len(df):,} email (Ham: {sum(df.label==0):,}, Spam: {sum(df.label==1):,})")

**Đúc kết từ làm sạch & khử trùng lặp:**
- **Bóc trần thủ thuật ngụy trang:** Khi hàm `clean_text` loại bỏ toàn bộ thẻ HTML và đường dẫn URL động, toàn bộ lớp ngụy trang đa hình của spammer bị lột sạch, phơi bày hàng nghìn email có nội dung hoàn toàn trùng khớp thuộc các chiến dịch phát tán thư rác hàng loạt.
- **Nguyên tắc Zero Duplicate Leakage:** Thao tác loại bỏ các mẫu trùng lặp nội dung văn bản sạch triệt tiêu hoàn toàn nguy cơ rò rỉ thông tin giữa tập huấn luyện và tập kiểm thử (Data Leakage), bảo đảm tính khách quan tuyệt đối cho kết quả đánh giá mô hình.

### 3.3 Lưu trữ Dữ liệu Sạch Đã Chuẩn hóa (`clean_emails.csv`)
**Mục tiêu:** Xuất 32.660 email sạch sang tệp CSV bàn giao cho Notebook 2.

In [ ]:
# Xuất tệp dữ liệu sạch
clean_path = os.path.join('..', 'data', 'processed', 'clean_emails.csv')
cols = ['sender', 'receiver', 'date', 'subject', 'body', 'text_clean', 'urls', 'label']
df[cols].to_csv(clean_path, index=False, encoding='utf-8')
print(f"Đã lưu {len(df):,} dòng sạch tại: {clean_path}")

**Tổng kết Giai đoạn 1:**  
Đã làm sạch toàn bộ dữ liệu thô, loại bỏ thư trùng lặp và lưu lại 32.660 email chất lượng cao vào tệp `clean_emails.csv`. Dữ liệu đã sẵn sàng để chuyển sang bước FE và huấn luyện mô hình ở Notebook tiếp theo.